# FLASH Soil Saturation Diagnostics
## Prototype Analysis Notebook (Single Case)
------

### Notebook Setup

In [6]:
# Let's import the needed libraries
import    urllib.request# helps us download files from the internet
import    gzip          # helps us unzip .gz files
import    tempfile      # creates a temporary file that will disappear later
import    xarray as xr  # the best tool to read weather raster files
from datetime import datetime # handles date and time
import numpy as np               # for arrays and math (np = nickname)
import matplotlib.pyplot as plt  # main plotting tool (plt = nickname)
import cartopy.crs as ccrs       # map projections
import cartopy.feature as cfeature  # add states, coastlines, etc.
import numpy.ma as ma            # mask/hide bad values
from metpy.plots import ctables  # official radar colors
import matplotlib.colors as mcolors

# Core scientific Python stack
import numpy as np
import pandas as pd
import rasterio
import rioxarray as rxr
from rasterio.enums import Resampling

# NLDAS Access
import earthaccess

# # Geospatial & plotting
# import geopandas as gpd
# import matplotlib.pyplot as plt
# import cartopy.crs as ccrs

# ML tools
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

# Image / segmentation utilities
from scipy import ndimage

### Step 1 - Define the Analysis Case
This section has three steps:

- Parse a string → datetime object
- Extract components from the datetime object
- Format those components back into strings for URLs, paths, or filenames

“We parse date‑time strings into Python’s datetime objects so we can extract and format year, month, day, and time components reliably when constructing file paths and URLs.”

In [41]:
# ===============================
# USER-DEFINED CONFIGURATION
# ===============================

case_date = "2024-06-22"
case_time = "19:00"  # UTC

# Simple bounding box (Iowa region)
lon_min, lon_max = -97.5, -90.0
lat_min, lat_max = 40.0, 45

# Parse it into a datetime object
case_dt = datetime.strptime(case_date + " " + case_time, "%Y-%m-%d %H:%M")

# Accessing Date Components Directly
year  = case_dt.year
month = case_dt.month
day   = case_dt.day
hour  = case_dt.hour
minute = case_dt.minute


### Step 2 — Download and Load Data
#### FLASH data from ISU Mesonet Archive (CREST Soil Saturation & Unit Q)

In [42]:
# A function that downloads FLASH data given a product name and a date
def downloadFLASH(flash_product,flash_date):
    year_str  = flash_date.strftime("%Y")   # "2019"
    month_str = flash_date.strftime("%m")   # "05"
    day_str   = flash_date.strftime("%d")   # "23"
    hour_str  = flash_date.strftime("%H")   # "12"
    minute_str = flash_date.strftime("%M")  # "00"
    
    url = (f"https://mtarchive.geol.iastate.edu/"
           f"{year_str}/{month_str}/{day_str}/mrms/ncep/FLASH/{flash_product}/"
           f"{flash_product}_00.00_{year_str}{month_str}{day_str}-{hour_str}0000.grib2.gz")

    response = urllib.request.urlopen(url)
    compressed_data = response.read()
    
    print(url)
    print(f"   → Got the file! Size: {len(compressed_data):,} bytes (compressed)")
    
    print("Starting to unzip and load the radar file...")
    
    # We create a temporary file on your computer.
    # This file will hold the unzipped version.
    with tempfile.NamedTemporaryFile(suffix=".grib2") as temp_file:
    
        # ────────────────
        # Part A: Unzip (decompress) the file we downloaded
        # ────────────────
        unzipped_data = gzip.decompress(compressed_data)
        
        print(f"   → Unzipped! New size = {len(unzipped_data):,} bytes")
    
        # Write the unzipped bytes into our temporary file
        temp_file.write(unzipped_data)
        
        # flush() = "make sure everything is really written to disk now"
        # (very important before reading the file)
        temp_file.flush()
        
        print(f"   → Unzipped file saved temporarily at: {temp_file.name}")
    
        # ────────────────
        # Part B: Read the weather data with xarray 
        # ────────────────
        data_in = xr.load_dataarray(
            temp_file.name,             # where the file is
            engine='cfgrib',            # special tool for weather GRIB files
            decode_timedelta=True       # helps with time information
        )

    # Process the downloaded data
    lons = data_in.longitude           # east-west positions (longitudes)
    lats = data_in.latitude            # north-south positions (latitudes)
    refl = data_in.values              # FLASH variable in each grid cell — the main data
    
    flash_layer = data_in.assign_coords(
        longitude=(((data_in.longitude + 180) % 360) - 180)
    ).sortby("longitude")
    
    flash_layer = flash_layer.rename({"longitude": "x", "latitude": "y"})
    flash_layer = flash_layer.rio.set_spatial_dims("x", "y")
    flash_layer = flash_layer.rio.write_crs("EPSG:4326")
    flash_layer = flash_layer.rio.write_nodata(-9999)
    flash_layer = flash_layer.rio.write_transform()

    return flash_layer

In [43]:
## FLASH UNIT Q Layer
flas_uq_da = downloadFLASH("CREST_MAXUNITSTREAMFLOW",case_dt)

## FLASH SOILSAT Layer
flas_soil_da = downloadFLASH("CREST_MAXSOILSAT",case_dt)

https://mtarchive.geol.iastate.edu/2024/06/22/mrms/ncep/FLASH/CREST_MAXUNITSTREAMFLOW/CREST_MAXUNITSTREAMFLOW_00.00_20240622-190000.grib2.gz
   → Got the file! Size: 2,018,453 bytes (compressed)
Starting to unzip and load the radar file...
   → Unzipped! New size = 2,086,663 bytes
   → Unzipped file saved temporarily at: /tmp/tmpegj0xho8.grib2
https://mtarchive.geol.iastate.edu/2024/06/22/mrms/ncep/FLASH/CREST_MAXSOILSAT/CREST_MAXSOILSAT_00.00_20240622-190000.grib2.gz
   → Got the file! Size: 3,848,678 bytes (compressed)
Starting to unzip and load the radar file...
   → Unzipped! New size = 3,928,842 bytes
   → Unzipped file saved temporarily at: /tmp/tmp4j7glkd5.grib2


#### NLDAS Layers
 Download outputs from the NOAH-MP Land Surface model - More information at https://ldas.gsfc.nasa.gov/nldas

In [44]:
# NLDAS

# Login (only once per machine)
earthaccess.login(persist=True)

# Download ONE real file
results = earthaccess.search_data(
    short_name="NLDAS_NOAH0125_H",
    version="2.0",
    temporal=case_dt.strftime("%Y-%m-%d %H:%M"), # ("2024-12-25 10:00", "2024-05-27 20:00"), #case_dt.strftime("%Y-%m-%d %H:%M"), # 
    count=1
)

print(f"Found {len(results)} granule")

files = earthaccess.download(results, "./nldas_data/")
local_file = files[0]
# print("File:", local_file)

# Open with xarray
ds_nldas = xr.open_dataset(local_file, engine="netcdf4")
# print(ds_nldas.variables)

# def dowloadNLDAS():
#     #https://hydro1.gesdisc.eosdis.nasa.gov/data/NLDAS/NLDAS_NOAH0125_H.2.0/2019/022/NLDAS_NOAH0125_H.A20190122.1500.020.nc

Found 1 granule


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

In [45]:
# Map NLDAS Soil layers onto FLASH Soil Saturation Definition

# Load in the Depth-to-Bedrock layer used in FLASH to determine Maximum Soil Depth (cm)
with rasterio.open("RockDepth_1km_mrms_grid.tif") as src:
    soil_depth = src.read(1)        # NumPy array
    transform = src.transform
    crs = src.crs
    nodata = src.nodata

# Load in the Effective Porosity layer used in FLASH to determine Maximum Water Capacity
with rasterio.open("Effective_Porosity_FLASH_MRMS_1km.tif") as src:
    soil_eff_porosity = src.read(1)        # NumPy array
    # transform = src.transform
    # crs = src.crs
    # nodata = src.nodata

# Resample and Reproject NLDAS to FLASH grid
# soil100cm_nldas = ds_nldas["SMAvail_0_200cm"]
soil100cm_nldas = ds_nldas["SoilM_0_200cm"]

soil100cm_nldas = soil100cm_nldas.rename({"lon": "x", "lat": "y"})

# Activate rioxarray
soil100cm_nldas = soil100cm_nldas.rio.set_spatial_dims(x_dim="x", y_dim="y")
soil100cm_nldas = soil100cm_nldas.rio.write_crs("EPSG:4326")
soil100cm_nldas = soil100cm_nldas.rio.write_transform()

# Reproject / resample
warped_soil100cm_nldas = soil100cm_nldas.rio.reproject(
    "EPSG:4326",
    resolution=0.01,
    resampling=Resampling.bilinear
    
)

# Get bounding box of reference raster
minx, miny, maxx, maxy = flas_uq_da.rio.bounds()

soil100cm_nldas_mrmsgrid = warped_soil100cm_nldas.rio.reproject_match(flas_uq_da)

# Water density
rho_water = 1000 # kg m-3

# Convert Soil moisture content kg m-2 to an estimate of Soil Saturation (%)
theta = soil100cm_nldas_mrmsgrid / (rho_water * soil_depth * 0.01) # theta = Volumetric soil moisture
nldas_soil100cm_flash_saturation = (theta / soil_eff_porosity) * 100


In [ ]:
# Create a 4-panel figure
fig, axs = plt.subplots(
    2, 2, figsize=(12, 8),
    subplot_kw={"projection": ccrs.PlateCarree()},
    constrained_layout=True
)

# Define general graphic settings
map_extent_for_graph = [lon_min, lon_max, lat_min, lat_max]

# FLASH PLOTTING FUNCTION
def plot_FLASH(c_ax,map_extent,flash_product,dataArray,sub_title):
    # Color - 
    all_products = {'unitq':{'label':r'Unit Q ($m^3/s.km^2$)', 'graphpars':{'clims':[0,20], 'clevs':[0,1,2,4,6,10,20], 'colors':['grey', 'lightgrey', 'yellow', 'orange', 'red', 'purple']}},
                'soilsat':{'label':r'Soil Saturation (%)', 'graphpars':{'clims':[0,100], 'clevs':[0,25,50,75,85,95,100], 'colors':['lightgrey', 'grey', 'green', 'yellow', 'orange', 'purple']}},
                   'nldas':{'label':r'Soil Moisture Content ($kg/m^2$)', 'graphpars':{'clims':[0,1000], 'clevs':[0,200,400,600,800,1000], 'colors':['lightgrey', 'grey', 'yellow', 'green', 'blue']}}}
    # FLASH UNIT Q
    ax = axs[c_ax[0], c_ax[1]]

    ax.set_extent(map_extent, crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.COASTLINE, linewidth=2)
    ax.add_feature(cfeature.BORDERS, linewidth=1)
    ax.add_feature(cfeature.STATES, linewidth=0.5)

    clevs = all_products[flash_product]['graphpars']['clevs']
    colors = all_products[flash_product]['graphpars']['colors']
    cmap = mcolors.ListedColormap(colors)
    norm = mcolors.BoundaryNorm(clevs, cmap.N)

    dataArray = dataArray.where(dataArray >= 0)
    
    # Create colored mesh
    mesh = dataArray.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        norm=norm,
        add_colorbar=False
    )

    # Add colorbar (legend) to explain colors
    cb = plt.colorbar(mesh, ax=ax, orientation='horizontal', pad=0.05, aspect=50)
    cb.set_label(all_products[flash_product]['label'])  # label explains what the numbers mean

    # Add title at the top
    ax.set_title(sub_title, fontsize=14)

# FLASH UNIT Q
current_ax = [0,0]
this_flash_product = 'unitq'
this_dataArray = flas_uq_da
plot_FLASH(current_ax,map_extent_for_graph,this_flash_product,this_dataArray, 'FLASH Maximum Unit Streamflow Layer')

# FLASH SOIL SATURATION
current_ax = [0,1]
this_flash_product = 'soilsat'
this_dataArray = flas_soil_da
plot_FLASH(current_ax,map_extent_for_graph,this_flash_product,this_dataArray, 'FLASH Maximum Soil Saturation Layer')

# NLDAS SOIL MOISTURE
current_ax = [1,0]
this_flash_product = 'nldas'
this_dataArray = soil100cm_nldas_mrmsgrid.isel(time=0)
plot_FLASH(current_ax,map_extent_for_graph,this_flash_product,this_dataArray, 'NLDAS Soil Moisture Content Layer')

# NLDAS SOIL SATURATION (FLASH Like)
current_ax = [1,1]
this_flash_product = 'soilsat'
this_dataArray = nldas_soil100cm_flash_saturation.isel(time=0)
plot_FLASH(current_ax,map_extent_for_graph,this_flash_product,this_dataArray, 'NLDAS FLASH-like Soil Saturation Layer')

# Show figure
plt.show()

In [ ]:
# # Step 3.1: Fix coordinate shape
# # Why? Some files give lons/lats as 1D lists (simple row or column)
# # But to plot, we need 2D grids (like a full table)
# if lons.ndim == 1 and lats.ndim == 1:
#     print("   → Coordinates were 1D lists → converting to 2D grid...")
#     # np.meshgrid = numpy tool that makes a full table of positions
#     # It combines the lon list and lat list into two matching 2D arrays
#     #################### Your code goes here ##############################
#     lons, lats = np.meshgrid(lons, lats)
#     print("   → Done! Now lons and lats are full 2D grids.")
# else:
#     print("   → Coordinates already 2D – good!")

# print("Coordinates and data extracted and ready for plotting!")

# # Plot the map
# # Create a new blank figure (using matplotlib: plt.figure())
# # fig = plt.figure(figsize=(10, 8))
# fig, axs = plt.subplots(2, 2, figsize=(8, 6))

# # Create the map drawing area (ax = our canvas)
# # ax = plt.axes(projection=ccrs.PlateCarree())
# axs[0, 0].plot(projection=ccrs.PlateCarree())

# # Zoom to your Region
# # ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
# axs[0, 0].set_extent(projection=ccrs.PlateCarree())

# # # Add basic geography outlines (from cartopy.feature)
# # ax.add_feature(cfeature.COASTLINE, linewidth=2)   # shorelines – thick line
# # ax.add_feature(cfeature.BORDERS, linewidth=1)     # country borders – thick line
# # ax.add_feature(cfeature.STATES, linewidth=0.5)    # US state lines – thinner line

# clevs  = [0, 1, 2, 4, 6, 10, 20]
# colors = ['grey', 'lightgrey', 'yellow', 'orange', 'red', 'purple']

# cmap = mcolors.ListedColormap(colors)
# norm = mcolors.BoundaryNorm(clevs, cmap.N)

# # Plot radar data as colored grid
# # mesh = ax.pcolormesh(
# #     lons, lats, ma.masked_where(refl < 0, refl),
# #     cmap=cmap,
# #     norm=norm,
# #     transform=ccrs.PlateCarree()
# # )
# mesh = axs[0, 0].pcolormesh(
#     lons, lats, ma.masked_where(refl < 0, refl),
#     cmap=cmap,
#     norm=norm,
#     transform=ccrs.PlateCarree()
# )

# # Add colorbar (legend) to explain colors
# cb = plt.colorbar(mesh, ax=ax, orientation='horizontal', pad=0.05, aspect=50)
# cb.set_label('Unit Streamflow (m^3/s.km^2)')  # label explains what the numbers mean

# # Add title at the top
# plt.title('FLASH Unit Streamflow Layer', fontsize=14)
# plt.show()

# # Plot
# fig = plt.figure(figsize=(10, 6))
# ax = plt.axes(projection=ccrs.PlateCarree())

# sm_vol.isel(time=0).plot(
#     ax=ax,
#     cmap="YlGnBu",
#     vmin=0,
#     vmax=0.45,
#     cbar_kwargs={"label": "Volumetric Soil Moisture (m³/m³)"}
# )

# ax.add_feature(cfeature.STATES, lw=0.4)
# ax.add_feature(cfeature.COASTLINE)
# ax.set_title("Top 10 cm Soil Moisture – " + dt.strftime("%Y%m%d.%H%M") + " UTC")
# ax.set_extent([-125, -66, 24, 50])

# plt.show()

## Concept Sidebar — Clustering

**Clustering** is an unsupervised learning technique used to identify natural groupings in data without predefined labels.
In this application, clustering helps us answer:
“Are there spatial regions where FLASH systematically differs from an independent soil moisture estimate?”
We are clustering soil saturation differences, not errors — interpretation requires hydrologic context.

In [ ]:
# Prepare data for clustering
X = soil_diff.values.reshape(-1, 1)
mask = np.isfinite(X[:, 0])

# K-means clustering
kmeans = KMeans(n_clusters=4, random_state=0)
labels = np.full(X.shape[0], np.nan)
labels[mask] = kmeans.fit_predict(X[mask])

cluster_map = labels.reshape(soil_diff.shape)

plt.imshow(cluster_map, cmap="tab10")
plt.title("Clusters of Soil Saturation Difference")
plt.colorbar(label="Cluster ID")
plt.show()